In [13]:
from scipy.io import loadmat

data = loadmat('datasets/imagelabels.mat')
print(data.keys())  # shows variable names inside
print(data['__header__'])  # access a specific variable
print(data['__version__'])  # access a specific variable
print(data['__globals__'])  # access a specific variable
labels = data['labels'][0]
 # access a specific variable
print(type(labels[0]))

dict_keys(['__header__', '__version__', '__globals__', 'labels'])
b'MATLAB 5.0 MAT-file, Platform: GLNX86, Created on: Thu Feb 19 15:43:33 2009'
1.0
[]
<class 'numpy.uint8'>


In [12]:
data = loadmat('datasets/setid.mat')
print(data.keys())  # shows variable names inside
print(data['__header__'])  # access a specific variable
print(data['__version__'])  # access a specific variable
print(data['__globals__'])  # access a specific variable
test = data['trnid'][0] # access a specific variable
valid = data['valid'][0] # access a specific variable
train = data['tstid'][0] # access a specific variable
print(data['tstid'][0][:10])  # access a specific variable


dict_keys(['__header__', '__version__', '__globals__', 'trnid', 'valid', 'tstid'])
b'MATLAB 5.0 MAT-file, Platform: GLNX86, Created on: Thu Feb 19 17:38:58 2009'
1.0
[]
[6734 6735 6737 6742 6743 6745 6746 6748 6751 6752]


In [9]:
import torch
ckpt = torch.load('pretrained_models/psp_flowers.pt', map_location='cpu')
print(ckpt.keys())

dict_keys(['state_dict', 'opts', 'latent_avg'])


In [15]:
import os
import shutil
from tqdm import tqdm

# Paths
root_dir = "proj"
src_dir = os.path.join(root_dir, "jpg")
target_dir = os.path.join(root_dir, "flowers")

# Your lists
# These should contain image IDs as integers or strings (e.g. "00001")
# Example:
# train_ids = ["00001", "00002", ...]
# val_ids = ["00010", "00011", ...]
# test_ids = ["00020", "00021", ...]
# class_ids = [...]  # same length as total images

# Make sure you have them defined in your notebook
assert len(labels) == len(os.listdir(src_dir)), "class_ids must match number of images!"

# Create split folders
splits = ["train", "valid", "test"]
for split in splits:
    os.makedirs(os.path.join(target_dir, split), exist_ok=True)

# Function to copy files
def copy_images(split_ids, split_name):
    split_dir = os.path.join(target_dir, split_name)
    for img_id in tqdm(split_ids, desc=f"Processing {split_name}"):
        img_name = f"image_{int(img_id):05d}.jpg"
        src_path = os.path.join(src_dir, img_name)

        # get class for this image
        idx = int(img_id) - 1  # adjust if indexing starts at 1
        class_label = str(labels[idx])

        # create class subfolder
        class_dir = os.path.join(split_dir, class_label)
        os.makedirs(class_dir, exist_ok=True)

        # copy image
        shutil.copy(src_path, os.path.join(class_dir, img_name))

# Run for each split
copy_images(train, "train")
copy_images(valid, "valid")
copy_images(test, "test")

print("✅ Dataset organized under", target_dir)


Processing train:   0%|          | 0/6149 [00:00<?, ?it/s]

Processing test: 100%|██████████| 1020/1020 [00:02<00:00, 379.95it/s]

✅ Dataset organized under proj/flowers


In [16]:
# save 10 png images into one image of 2 rows and 5 columns
from PIL import Image
import numpy as np
import os

def save_image_grid(image_paths, grid_size, save_path):
    images = [Image.open(img_path) for img_path in image_paths]
    widths, heights = zip(*(i.size for i in images))

    max_width = max(widths)
    max_height = max(heights)

    grid_width = grid_size[1] * max_width
    grid_height = grid_size[0] * max_height

    new_im = Image.new('RGB', (grid_width, grid_height))

    for idx, im in enumerate(images):
        row = idx // grid_size[1]
        col = idx % grid_size[1]
        new_im.paste(im, (col * max_width, row * max_height))

    new_im.save(save_path)

image_paths = [f"/home/fvaleau/HAE/traj_images/animals_last/sample_{i+1}.png" for i in range(8)]
save_image_grid(image_paths, (8, 1), "animals_last.png")

In [8]:
id =  [0.0, 4.3, 18.2, 50.6, 111.2, 218.8, 393.4, 665.7, 928.0, 1000.0]
a = [int(1000-x) for x in id]
print(list(reversed(a)))

[0, 72, 334, 606, 781, 888, 949, 981, 995, 1000]


In [11]:
import torch
from geoopt.manifolds import PoincareBall
embed = torch.load('/home/fvaleau/HAE/Visualization/trajs_from0.3', map_location=torch.device('cpu'))
print(embed.shape)
dist = PoincareBall.dist(PoincareBall(),x=embed[1,-1,:], y=embed[-1,-1,:])
dist1 = PoincareBall.dist(PoincareBall(),x=embed[1,-1,:], y=embed[2,-1,:])
dist2 = PoincareBall.dist(PoincareBall(),x=embed[-1,-1,:], y=embed[2,-1,:])


for i in range(7):
    print(PoincareBall.dist(PoincareBall(),x=embed[0,-1,:], y=embed[i+1,-1,:]))
print(dist, dist1, dist2)


torch.Size([8, 11, 512])
tensor(11.7489)
tensor(11.7192)
tensor(11.7835)
tensor(11.8032)
tensor(11.7642)
tensor(11.6975)
tensor(11.7489)
tensor(11.7642) tensor(11.7874) tensor(11.6832)


/scratch-local/75993/ipykernel_409769/3969906170.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  embed = torch.load('/home/fvaleau/HAE/Visualization/trajs_from0.3', map_

In [14]:
import pickle
import torch

with open("proj/animal_embedding/animalfaces_embed/archive/data.pkl", "rb") as f:
    data = pickle.load(f)

print(len(data), type(data))
print(type(data[0]))

#data = torch.tensor(data)
torch.save(data, "proj/animal_embedding/label")
print(type(data[0]))

7448 <class 'list'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [1]:
import torch
# checking embedding dimensionality
embd = torch.load("proj/umap_flowers/embed")
embed_all = torch.load("proj/umap_flowers/embed_all")
embed_hyper = torch.load("proj/umap_flowers/embed_hyper")
euc_embed = torch.load("proj/umap_flowers/euc_embed")

embd_t = torch.load("proj/umap_flowers_train/embed")
euc_embed_t = torch.load("proj/umap_flowers_train/euc_embed")

embd_v = torch.load("proj/umap_flowers_val/embed")
euc_embed_v = torch.load("proj/umap_flowers_val/euc_embed")

print("embed",len(embd), embd[0].shape)
print("embed_all",len(embed_all), embed_all[0].shape)
print("embed_hyper",len(embed_hyper), embed_hyper[0].shape)
print("euc_embed",len(euc_embed), euc_embed[0].shape)
print("embd_t",len(embd_t), embd_t[0].shape)
print("euc_embed_t",len(euc_embed_t), euc_embed_t[0].shape)
print("embd_v",len(embd_v), embd_v[0].shape)
print("euc_embed_v",len(euc_embed_v), euc_embed_v[0].shape)

/scratch-local/75993/ipykernel_790124/4181857088.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  embd = torch.load("proj/umap_flowers/embed")
/scratch-local/75993/ipyker

embed 1020 (512,)
embed_all 8189 (512,)
embed_hyper 1020 torch.Size([512])
euc_embed 1020 (18, 512)
embd_t 6149 (512,)
euc_embed_t 6149 (18, 512)
embd_v 1020 (512,)
euc_embed_v 1020 (18, 512)


In [8]:
euc_embd_tv = torch.cat((torch.tensor(euc_embed_t), torch.tensor(euc_embed_v)), dim=0)
euc_embd_tv_flat= torch.flatten(euc_embd_tv, start_dim=1   )
print(euc_embd_tv.shape, euc_embd_tv_flat.shape)
torch.save(euc_embd_tv_flat, "proj/umap_flowers/euc_embed_tv")



torch.Size([7169, 18, 512]) torch.Size([7169, 9216])


In [11]:
print(euc_embd_tv_flat.std())
print(euc_embd_tv_flat.mean())

tensor(0.3750)
tensor(0.2353)


In [4]:
import torch
from Visualization.notebook_utils.pmath import dist0

embd = torch.load("proj/flowers_emb_0.5/embed")
for i in range(10):
    r = dist0(torch.tensor(embd[i]), c=3)      # uncomment
    print(r)


/scratch-local/75993/ipykernel_2780057/3216934709.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  embd = torch.load("proj/flowers_emb_0.5/embed")


tensor(7.0464)
tensor(7.0464)
tensor(7.0464)
tensor(7.0464)
tensor(7.0464)
tensor(7.0464)
tensor(7.0464)
tensor(7.0464)
tensor(7.0464)
tensor(7.0464)
